# Day 00 · 딥러닝 기초 실습 (Colab GPU)
**Fundamentals of Deep Learning** — 신경망 → CNN → 데이터 증강 → 전이학습

> ⚙️ **먼저 GPU 켜기**: 상단 메뉴 **런타임 → 런타임 유형 변경 → 하드웨어 가속기 = GPU(T4)** 선택 후 저장.

이 노트북은 PyTorch로 이미지 분류 모델을 직접 학습시키며, MLP → CNN → 증강 → 전이학습으로 정확도를 끌어올립니다.


## Lab 0 · GPU 확인
`cuda`가 출력되면 준비 완료.


In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
print("GPU:", torch.cuda.get_device_name(0) if device == "cuda" else "CPU (런타임에서 GPU를 켜세요)")


## Lab 1 · 데이터 불러오기 (FashionMNIST)
28×28 흑백 의류 이미지 10종, 6만 장.


In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

tf = transforms.ToTensor()
train_set = datasets.FashionMNIST("./data", train=True,  download=True, transform=tf)
test_set  = datasets.FashionMNIST("./data", train=False, download=True, transform=tf)

train_dl = DataLoader(train_set, batch_size=64,  shuffle=True)
test_dl  = DataLoader(test_set,  batch_size=256)
classes = train_set.classes
print(len(train_set), "학습 이미지 |", len(test_set), "테스트 |", len(classes), "클래스")


In [ ]:
# 샘플 몇 장 살펴보기
import matplotlib.pyplot as plt
imgs, labels = next(iter(train_dl))
fig, ax = plt.subplots(1, 6, figsize=(12, 2.2))
for i in range(6):
    ax[i].imshow(imgs[i][0], cmap="gray"); ax[i].set_title(classes[labels[i]], fontsize=9); ax[i].axis("off")
plt.show()


## Lab 2 · 작은 신경망(MLP) 정의
입력 784 → 은닉 128(ReLU) → 출력 10.


In [ ]:
import torch.nn as nn

def make_mlp():
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, 128), nn.ReLU(),
        nn.Linear(128, 10),
    )

mlp = make_mlp().to(device)
print(mlp)


## Lab 3 · 학습 루프 (예측→손실→역전파→수정)
재사용할 수 있도록 함수로 만듭니다.


In [ ]:
def train(model, dl, epochs=3, lr=1e-3):
    model.train()
    loss_fn = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        total = 0.0
        for x, y in dl:
            x, y = x.to(device), y.to(device)
            pred = model(x)                 # ① 순전파
            loss = loss_fn(pred, y)         # ② 손실
            opt.zero_grad(); loss.backward()# ③ 역전파
            opt.step()                      # ④ 수정
            total += loss.item()
        print(f"epoch {epoch+1}: loss={total/len(dl):.4f}")
    return model

@torch.no_grad()
def accuracy(model, dl):
    model.eval(); correct = total = 0
    for x, y in dl:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(1) == y).sum().item(); total += y.size(0)
    return correct / total


In [ ]:
train(mlp, train_dl, epochs=3)
print("MLP 테스트 정확도:", round(accuracy(mlp, test_dl), 4))


## Lab 5 · CNN으로 업그레이드
conv→pool→conv→pool→FC. 같은 학습 함수로 재학습 → 정확도 비교.


In [ ]:
def make_cnn():
    return nn.Sequential(
        nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 28→14
        nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 14→7
        nn.Flatten(),
        nn.Linear(64*7*7, 128), nn.ReLU(),
        nn.Linear(128, 10),
    )

cnn = make_cnn().to(device)
train(cnn, train_dl, epochs=3)
print("CNN 테스트 정확도:", round(accuracy(cnn, test_dl), 4))   # MLP보다 높아지는지 확인


## Lab 6 · 데이터 증강
좌우반전·회전을 추가해 재학습 → 과적합이 줄어드는지(정확도 변화) 확인.


In [ ]:
aug = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
])
train_aug = datasets.FashionMNIST("./data", train=True, download=True, transform=aug)
train_aug_dl = DataLoader(train_aug, batch_size=64, shuffle=True)

cnn_aug = make_cnn().to(device)
train(cnn_aug, train_aug_dl, epochs=3)
print("증강 CNN 테스트 정확도:", round(accuracy(cnn_aug, test_dl), 4))


## Lab 7 · 전이학습 (사전학습 ResNet18)
ImageNet으로 학습된 ResNet의 '보는 법'을 재사용하고, **마지막 분류층만** 새로 학습합니다.
FashionMNIST를 3채널·224로 맞추고, 빠른 데모를 위해 **일부 데이터·1 epoch**만 사용합니다.


In [ ]:
from torchvision import models
from torch.utils.data import Subset

tf_resnet = transforms.Compose([
    transforms.Resize(224),
    transforms.Grayscale(3),       # 흑백 → 3채널
    transforms.ToTensor(),
])
big_train = datasets.FashionMNIST("./data", train=True,  download=True, transform=tf_resnet)
big_test  = datasets.FashionMNIST("./data", train=False, download=True, transform=tf_resnet)
# 데모용 부분집합 (속도)
sub_train = DataLoader(Subset(big_train, range(3000)), batch_size=64, shuffle=True)
sub_test  = DataLoader(Subset(big_test,  range(1000)), batch_size=128)

net = models.resnet18(weights="IMAGENET1K_V1")
for p in net.parameters():            # 사전학습 특징 얼리기
    p.requires_grad = False
net.fc = nn.Linear(net.fc.in_features, 10)   # 마지막 층만 새로
net = net.to(device)

train(net, sub_train, epochs=1, lr=1e-3)      # 마지막 층만 학습
print("전이학습 테스트 정확도(부분집합):", round(accuracy(net, sub_test), 4))


## Lab 8 · 도전 (선택)
- **A** 하이퍼파라미터: `lr`·`batch_size`·은닉 크기를 바꿔 최고 정확도 갱신
- **B** 오분류 살펴보기: 틀린 이미지를 출력해 '왜 틀렸나' 관찰
- **C** CIFAR-10(컬러)으로 같은 파이프라인 적용

> 📝 **산출물**: MLP·CNN·증강·전이학습의 정확도를 표로 비교해 보세요.


In [ ]:
# 도전 B 예시 — 틀린 예측 살펴보기
import torch
cnn.eval()
x, y = next(iter(test_dl)); x, y = x.to(device), y.to(device)
pred = cnn(x).argmax(1)
wrong = (pred != y).nonzero(as_tuple=True)[0][:6]
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, len(wrong), figsize=(12, 2.2))
for k, idx in enumerate(wrong):
    ax[k].imshow(x[idx][0].cpu(), cmap="gray")
    ax[k].set_title(f"예측 {classes[pred[idx]]}\n정답 {classes[y[idx]]}", fontsize=8); ax[k].axis("off")
plt.show()
